# 5 · The southern hemisphere

Custom parameters: running the chain in the southern hemisphere.

`STANDARD_PARAMS` carries Van Wagner's day-length factors for 46°N,
January first. South of the equator the seasons are flipped, so the common
approximation is to shift both tables by six months. For latitudes far from
46° (north or south), use tables adjusted for your band, such as the ones
the `cffdrs` R package ships.

The same code as [`examples/05_southern_hemisphere.py`](../examples/05_southern_hemisphere.py).

> **A fire weather index is not an alert.** It says how easily a fire would start and spread in the weather given, not that one will. The official fire danger for Europe is published by [EFFIS](https://forest-fire.emergency.copernicus.eu/).

In [1]:
from fwi_chain import STANDARD_PARAMS, FwiParams, advance

In [2]:
def shift_six_months(table: tuple[float, ...]) -> tuple[float, ...]:
    return table[6:] + table[:6]

In [3]:
SOUTHERN = FwiParams(
    ffmc_start=85.0,
    dmc_start=6.0,
    dc_start=15.0,
    day_length_dmc=shift_six_months(STANDARD_PARAMS.day_length_dmc),
    day_length_dc=shift_six_months(STANDARD_PARAMS.day_length_dc),
)

# The same hot January afternoon, north and south of the equator.
for label, params in (("46°N tables", STANDARD_PARAMS), ("shifted (south)", SOUTHERN)):
    out = advance(
        params.initial_state,
        month=1,
        temperature_c=30.0,
        relative_humidity_pct=30.0,
        wind_speed_kmh=20.0,
        rain_24h_mm=0.0,
        params=params,
    )
    print(f"{label:<16} DMC {out.state.dmc:5.2f}  DC {out.state.dc:5.2f}  FWI {out.fwi:5.2f}")

# A table with the wrong number of months is refused up front.
try:
    FwiParams(85.0, 6.0, 15.0, day_length_dmc=(1.0,) * 11, day_length_dc=(0.0,) * 12)
except ValueError as e:
    print(f"Refused: {e}")

46°N tables      DMC  8.68  DC 20.10  FWI 14.09
shifted (south)  DMC 11.11  DC 24.10  FWI 15.59
Refused: day_length_dmc must have 12 monthly entries, got 11
